[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/state-reducers.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239428-lesson-2-state-reducers)

# 状态Reducer

## 回顾

我们学习了定义LangGraph状态模式的几种不同方法，包括`TypedDict`、`Pydantic`或`Dataclasses`。
 
## 目标

现在，我们将深入了解reducer，它们指定如何对状态模式中的特定键/通道执行状态更新。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph

## 默认覆写状态

让我们使用`TypedDict`作为我们的状态模式。

In [ ]:
from typing_extensions import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    foo: int

def node_1(state):
    print("---节点 1---")
    return {"foo": state['foo'] + 1}

# 构建图
builder = StateGraph(State)
builder.add_node("node_1", node_1)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"foo" : 1})

让我们看一下状态更新，`return {"foo": state['foo'] + 1}`。

如前所述，默认情况下LangGraph不知道更新状态的首选方式。
 
所以，它将只是覆盖`node_1`中`foo`的值：

```
return {"foo": state['foo'] + 1}
```
 
如果我们传递`{'foo': 1}`作为输入，从图返回的状态是`{'foo': 2}`。

## 分支

让我们看一个节点分支的情况。

In [ ]:
class State(TypedDict):
    foo: int

def node_1(state):
    print("---节点 1---")
    return {"foo": state['foo'] + 1}

def node_2(state):
    print("---节点 2---")
    return {"foo": state['foo'] + 1}

def node_3(state):
    print("---节点 3---")
    return {"foo": state['foo'] + 1}

# 构建图
builder = StateGraph(State)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_1", "node_3")
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langgraph.errors import InvalidUpdateError
try:
    graph.invoke({"foo" : 1})
except InvalidUpdateError as e:
    print(f"发生InvalidUpdateError: {e}")

我们看到了一个问题！

节点1分支到节点2和3。

节点2和3并行运行，这意味着它们在图的同一步骤中运行。

它们都试图*在同一步骤内*覆盖状态。

这对图来说是模糊的！它应该保留哪个状态？

## Reducer

[Reducer](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers)为我们提供了解决这个问题的通用方法。

它们指定如何执行更新。

我们可以使用`Annotated`类型来指定reducer函数。

例如，在这种情况下，让我们将每个节点返回的值追加而不是覆盖它们。

我们只需要一个可以执行此操作的reducer：`operator.add`是Python内置operator模块中的一个函数。

当`operator.add`应用于列表时，它执行列表连接。

In [ ]:
from operator import add
from typing import Annotated

class State(TypedDict):
    foo: Annotated[list[int], add]

def node_1(state):
    print("---节点 1---")
    return {"foo": [state['foo'][0] + 1]}

# 构建图
builder = StateGraph(State)
builder.add_node("node_1", node_1)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"foo" : [1]})

现在，我们的状态键`foo`是一个列表。

这个`operator.add` reducer函数将把每个节点的更新追加到这个列表中。

In [ ]:
def node_1(state):
    print("---节点 1---")
    return {"foo": [state['foo'][-1] + 1]}

def node_2(state):
    print("---节点 2---")
    return {"foo": [state['foo'][-1] + 1]}

def node_3(state):
    print("---节点 3---")
    return {"foo": [state['foo'][-1] + 1]}

# 构建图
builder = StateGraph(State)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_1", "node_3")
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

我们可以看到节点2和3中的更新是并发执行的，因为它们在同一步骤中。

In [ ]:
graph.invoke({"foo" : [1]})

现在，让我们看看如果我们向`foo`传递`None`会发生什么。

我们看到一个错误，因为我们的reducer `operator.add`试图将作为输入传递的`NoneType`与`node_1`中的列表连接。

In [ ]:
try:
    graph.invoke({"foo" : None})
except TypeError as e:
    print(f"发生TypeError: {e}")

## 自定义Reducer

为了解决这样的情况，[我们还可以定义自定义reducer](https://langchain-ai.github.io/langgraph/how-tos/subgraph/#custom-reducer-functions-to-manage-state)。

例如，让我们定义自定义reducer逻辑来组合列表并处理任一或两个输入可能为`None`的情况。

In [ ]:
def reduce_list(left: list | None, right: list | None) -> list:
    """安全地组合两个列表，处理任一或两个输入可能为None的情况。

    Args:
        left (list | None): 要组合的第一个列表，或None。
        right (list | None): 要组合的第二个列表，或None。

    Returns:
        list: 包含两个输入列表所有元素的新列表。
               如果输入为None，则将其视为空列表。
    """
    if not left:
        left = []
    if not right:
        right = []
    return left + right

class DefaultState(TypedDict):
    foo: Annotated[list[int], add]

class CustomReducerState(TypedDict):
    foo: Annotated[list[int], reduce_list]

在`node_1`中，我们追加值2。

In [ ]:
def node_1(state):
    print("---节点 1---")
    return {"foo": [2]}

# 构建图
builder = StateGraph(DefaultState)
builder.add_node("node_1", node_1)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

try:
    print(graph.invoke({"foo" : None}))
except TypeError as e:
    print(f"发生TypeError: {e}")

现在，尝试使用我们的自定义reducer。我们可以看到没有抛出错误。

In [ ]:
# 构建图
builder = StateGraph(CustomReducerState)
builder.add_node("node_1", node_1)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

try:
    print(graph.invoke({"foo" : None}))
except TypeError as e:
    print(f"发生TypeError: {e}")

## 消息

在模块1中，我们展示了如何使用内置reducer `add_messages`来处理状态中的消息。

我们还展示了[`MessagesState`是一个有用的快捷方式，如果你想使用消息](https://langchain-ai.github.io/langgraph/concepts/low_level/#messagesstate)。

* `MessagesState`有一个内置的`messages`键
* 它还为这个键有一个内置的`add_messages` reducer

这两者是等价的。

为了简洁，我们将通过`from langgraph.graph import MessagesState`使用`MessagesState`类。

In [ ]:
from typing import Annotated
from langgraph.graph import MessagesState
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

# 定义一个包含带有add_messages reducer的消息列表的自定义TypedDict
class CustomMessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    added_key_1: str
    added_key_2: str
    # 等等

# 使用MessagesState，它包括带有add_messages reducer的messages键
class ExtendedMessagesState(MessagesState):
    # 添加除了预构建的messages之外需要的任何键
    added_key_1: str
    added_key_2: str
    # 等等

让我们更多地讨论`add_messages` reducer的使用。

In [ ]:
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, HumanMessage

# 初始状态
initial_messages = [AIMessage(content="你好！我如何能协助你？", name="Model"),
                    HumanMessage(content="我正在寻找有关海洋生物学的信息。", name="Lance")
                   ]

# 要添加的新消息
new_message = AIMessage(content="当然，我可以帮助你。你具体对什么感兴趣？", name="Model")

# 测试
add_messages(initial_messages , new_message)

所以我们可以看到`add_messages`允许我们将消息追加到状态中的`messages`键。

### 重写

让我们展示一些在使用`add_messages` reducer时的有用技巧。

如果我们传递一个与`messages`列表中现有消息具有相同ID的消息，它将被覆盖！

In [ ]:
# 初始状态
initial_messages = [AIMessage(content="你好！我如何能协助你？", name="Model", id="1"),
                    HumanMessage(content="我正在寻找有关海洋生物学的信息。", name="Lance", id="2")
                   ]

# 要添加的新消息
new_message = HumanMessage(content="我正在寻找有关鲸鱼的信息，具体来说", name="Lance", id="2")

# 测试
add_messages(initial_messages , new_message)

### 删除

`add_messages`还[支持消息删除](https://langchain-ai.github.io/langgraph/how-tos/memory/delete-messages/)。

为此，我们只需使用`langchain_core`中的[RemoveMessage](https://api.python.langchain.com/en/latest/messages/langchain_core.messages.modifier.RemoveMessage.html)。

In [ ]:
from langchain_core.messages import RemoveMessage

# 消息列表
messages = [AIMessage("嗨。", name="Bot", id="1")]
messages.append(HumanMessage("嗨。", name="Lance", id="2"))
messages.append(AIMessage("所以你说你在研究海洋哺乳动物？", name="Bot", id="3"))
messages.append(HumanMessage("是的，我了解鲸鱼。但是我还应该了解其他什么？", name="Lance", id="4"))

# 隔离要删除的消息
delete_messages = [RemoveMessage(id=m.id) for m in messages[:-2]]
print(delete_messages)

In [ ]:
add_messages(messages , delete_messages)

我们可以看到，如`delete_messages`中所述的消息ID 1和2被reducer删除了。

我们稍后会看到这个实际应用。